In [1]:
import scanpy as sc
import pandas as pd
import numpy as np
from scipy import sparse

sc.settings.verbosity = 3
sc.set_figure_params(dpi=200)

In [81]:
# Paths

HNOCA_PATH = "/Users/nickmahtani/Documents/1 - Projects/Jain Lab/Mechanotransduction/data/hnoca_cleanedmeta.h5ad"
ECM_HNOCA_PATH = "/Users/nickmahtani/Documents/1 - Projects/Jain Lab/Mechanotransduction/data/adata_ecm.h5ad"
ECM_XLSX_PATH = "/Users/nickmahtani/Documents/1 - Projects/Jain Lab/Mechanotransduction/data/ECM_Gene_Set_Curated.xlsx"
OUTPUT_DIR = "/Users/nickmahtani/Documents/1 - Projects/Jain Lab/Mechanotransduction/data/ecm_filtering_output"

# Thresholds

MIN_PCT_CELLS = 1.0         # To collect genes that appear in >= 1% of all the cells
MIN_PCT_CELLS_CELLTYPE = 5.0        # To collect genes that appear in >= 5% of at least 1 cell type

# Cell type annotations
CELLTYPE_COL = "annot_level_2"
REGION_COL = "annot_region_rev2"
AGE_COL = "organoid_age_days"

### 1. Loading the ECM gene list that I curated

In [3]:
# Importing into a pandas dataset

ecm_df = pd.read_excel(ECM_XLSX_PATH, sheet_name="Master Gene List")
ecm_df.head()


,Gene Symbol,Gene Name,Category,Subcategory,Matrisome Division,Matrisome Category,Key Functions / Notes
0,COL1A1,Collagen Type I Alpha 1 Chain,Collagens,Fibrillar,Core Matrisome,Collagens,"Major structural collagen; bone, skin, tendon"
1,COL1A2,Collagen Type I Alpha 2 Chain,Collagens,Fibrillar,Core Matrisome,Collagens,Partners with COL1A1 in type I collagen
2,COL2A1,Collagen Type II Alpha 1 Chain,Collagens,Fibrillar,Core Matrisome,Collagens,Primary cartilage collagen
3,COL3A1,Collagen Type III Alpha 1 Chain,Collagens,Fibrillar,Core Matrisome,Collagens,"Skin, blood vessels; co-distributes with type I"
4,COL4A1,Collagen Type IV Alpha 1 Chain,Collagens,Basement Membrane / Network,Core Matrisome,Collagens,Basement membrane network collagen


In [4]:
# Creating a python list of just the gene names

ecm_genes = ecm_df["Gene Symbol"].unique().tolist()
print(f"Number of unique genes: {len(ecm_genes)}")

Number of unique genes: 259


### 2. Loading the HNOCA atlas and matching genes with my list

In [10]:
# Loading the HNOCA atlas (in backed mode to save ram)

adata = sc.read_h5ad(HNOCA_PATH, backed="r")
print(f"Cells: {adata.n_obs:,} x Genes: {adata.n_vars}")

Cells: 1,770,578 x Genes: 36842


In [11]:
# Matching genes using sets

atlas_genes = set(adata.var_names)
found = [g for g in ecm_genes if g in atlas_genes]
missing = [g for g in ecm_genes if g not in atlas_genes]        # just curious

print(f"ECM genes found in atlas: {len(found)}")
if missing:
        print(f"ECM genes not found in atlas: {len(missing)}")
        print(missing)
 


ECM genes found in atlas: 257
ECM genes not found in atlas: 2
['MKL1', 'MKL2']


### 3. Subsetting genes to just ECM genes to save ram

In [9]:
adata_ecm = adata[:, found]  # perhaps i save this as another object to run analysis if ram ocntinues to be an issue
adata_ecm.write_h5ad(ECM_HNOCA_PATH)

#adata_ecm.n_vars

In [5]:
# Start from here to save ram
adata_ecm = sc.read_h5ad(ECM_HNOCA_PATH)


In [6]:
print(f"Cells: {adata_ecm.n_obs:,} x Genes: {adata_ecm.n_vars}")

# To see how sparse/dense the matrix is:

print(f"Matrix density: {(adata_ecm.X.nnz / np.prod(adata_ecm.shape))*100:.1f}%")

Cells: 1,770,578 x Genes: 257
Matrix density: 8.6%


### 4. Global Expression Statistics

In [7]:
# Computing percentage of cells that express each of the 257 genes

X= adata_ecm.X
X_bool = X>0 
pct_cells = np.array(X_bool.mean(axis=0)).flatten()*100
pct_cells[:10]


array([ 3.15919434,  6.02475576,  7.31546422,  2.30100001,  7.46033216,
       15.49482711,  0.26731384,  0.35835755, 13.21517606, 10.54175529])

In [8]:
# Computing mean expression of the genes across the different cells:

mean_expr = np.array(X.mean(axis=0)).flatten()
mean_expr[:10]

array([0.18655562, 0.34797454, 0.37816826, 0.14430663, 0.3690629 ,
       0.78863513, 0.01317454, 0.01725916, 0.6866173 , 0.5392851 ],
      dtype=float32)

In [12]:
# I wonder if these ECM genes were originally labeled as highly variable in the original analysis. 

hvg_status = adata.var.loc[found, "highly_variable"].values
print(f"{hvg_status.sum()} of the {len(found)} genes are higly variable ({hvg_status.mean() *100:.1f}%)")

75 of the 257 genes are higly variable (29.2%)


In [46]:
# Building results table:

results = pd.DataFrame({
    "gene_symbol": found,
    "pct_cells_expressing": pct_cells,
    "mean expression": mean_expr,
    "is_hvg_in_atlas": hvg_status
})
results.head()

,gene_symbol,pct_cells_expressing,mean expression,is_hvg_in_atlas
0,COL1A1,3.159194,0.186556,True
1,COL1A2,6.024756,0.347975,True
2,COL2A1,7.315464,0.378168,True
3,COL3A1,2.301000,0.144307,True
4,COL4A1,7.460332,0.369063,True


In [47]:
# Merging this table with the annotations I created in the ECM database

results = results.merge(
    ecm_df[["Gene Symbol", "Category", "Subcategory", "Matrisome Division"]].drop_duplicates(subset="Gene Symbol", keep="first"),
    left_on="gene_symbol",
    right_on="Gene Symbol",
    how="left"
).drop(columns="Gene Symbol")
results = results.sort_values("pct_cells_expressing", ascending=False)  #ordereded by extent of expression in cells

# Flagging genes that are very lowly expressed (<1% of cells)

results["passes_global_filter"] = results["pct_cells_expressing"] >= MIN_PCT_CELLS

results.head()

,gene_symbol,pct_cells_expressing,mean expression,is_hvg_in_atlas,Category,Subcategory,Matrisome Division,passes_global_filter
246,ACTB,97.233163,7.684915,False,Mechanotransduction,Cytoskeletal,N/A (Intracellular),True
184,RPSA,89.359463,6.517297,False,ECM Receptors,Laminin Receptors,Matrisome-Associated,True
255,RAC1,81.864679,5.163778,False,Mechanotransduction,Rho GTPase Signaling,N/A (Intracellular),True
256,CDC42,73.255061,4.454232,False,Mechanotransduction,Rho GTPase Signaling,N/A (Intracellular),True
252,RHOA,58.075386,3.310714,False,Mechanotransduction,Rho GTPase Signaling,N/A (Intracellular),True


In [48]:
print(f"  Genes ≥{MIN_PCT_CELLS}% cells: {results['passes_global_filter'].sum()}/{len(results)}")
print(f"  Genes in atlas HVG set: {results['is_hvg_in_atlas'].sum()}/{len(results)}")

  Genes ≥1.0% cells: 184/257
  Genes in atlas HVG set: 75/257


In [15]:
# if only want o deal with hvg

hvg_results = results[results["is_hvg_in_atlas"]].drop(columns="is_hvg_in_atlas")
hvg_results.head()

,gene_symbol,pct_cells_expressing,mean expression,Category,Subcategory,Matrisome Division
243,SYNE2,47.201535,2.795328,Mechanotransduction,LINC Complex,N/A (Nuclear Envelope)
58,VCAN,41.686726,2.397927,Proteoglycans,Lectican / CSPG,Core Matrisome
95,SPARC,29.346688,1.665615,ECM Glycoproteins,Core Glycoprotein,Core Matrisome
239,LMNB1,26.028901,1.393774,Mechanotransduction,Nuclear Mechanotransduction,N/A (Nuclear)
207,TIMP1,25.604577,1.413863,ECM Regulators,TIMPs,Matrisome-Associated


### 5. Which cell types produce what?

In [16]:
# Extracting cell types
celltypes = adata_ecm.obs[CELLTYPE_COL].astype(str)
unique_cts = sorted(celltypes.unique())
print(f"  Cell types (level 2): {len(unique_cts)}")

  Cell types (level 2): 17


In [17]:
celltypes

homosapiens_hindbrain_2020_bdrhapsodywholetranscriptomeanalysis_andersenjimena_001_d10_1016_j_cell_2020_11_017_53        Non-telencephalic NPC
homosapiens_hindbrain_2020_bdrhapsodywholetranscriptomeanalysis_andersenjimena_001_d10_1016_j_cell_2020_11_017_69        Non-telencephalic NPC
homosapiens_hindbrain_2020_bdrhapsodywholetranscriptomeanalysis_andersenjimena_001_d10_1016_j_cell_2020_11_017_72     Non-telencephalic Neuron
homosapiens_hindbrain_2020_bdrhapsodywholetranscriptomeanalysis_andersenjimena_001_d10_1016_j_cell_2020_11_017_76        Non-telencephalic NPC
homosapiens_hindbrain_2020_bdrhapsodywholetranscriptomeanalysis_andersenjimena_001_d10_1016_j_cell_2020_11_017_78        Non-telencephalic NPC
                                                                                                                               ...            
homosapiens_cerebralcortex_2017_smartseq2_sloansteven_001_d10_1016_j_neuron_2017_07_035_706                          Ventral Telencephalic NPC

In [18]:
# perhaps dont use found and use the HVG only?

ct_means = pd.DataFrame(index=found, columns=unique_cts, dtype=float)
ct_pct = pd.DataFrame(index=found, columns=unique_cts, dtype=float)
 
for ct in unique_cts:
    mask = (celltypes == ct).values
    n_cells = mask.sum()
    if n_cells == 0:  
        continue
    X_ct = X[mask, :]

    ct_means[ct] = np.array(X_ct.mean(axis=0)).flatten()
    ct_pct[ct] = np.array((X_ct > 0).mean(axis=0)).flatten() * 100

In [ ]:
# Mean gene expression of the different cell types per gene
ct_means

,Astrocyte,CP,Dorsal Telencephalic IP,Dorsal Telencephalic NPC,Dorsal Telencephalic Neuron,EC,Glioblast,MC,Microglia,NC Derivatives,Neuroepithelium,Non-telencephalic NPC,Non-telencephalic Neuron,OPC,PSC,Ventral Telencephalic NPC,Ventral Telencephalic Neuron
COL1A1,0.063804,0.190792,0.021751,0.031475,0.033866,0.032883,0.060157,2.593431,0.482706,2.380942,0.359064,0.268615,0.097504,0.057168,1.158075,0.041461,0.019693
COL1A2,0.374720,0.674918,0.036056,0.064129,0.020300,0.000000,0.215229,3.175522,0.868481,3.900692,0.872645,0.630214,0.159682,0.121693,2.368369,0.233032,0.110339
COL2A1,0.010661,0.478036,0.084631,0.425002,0.007851,0.009805,0.025597,0.248180,0.000000,0.673615,3.082153,0.602628,0.070926,0.095163,4.678299,0.917264,0.023228
COL3A1,0.114932,0.143528,0.014027,0.011593,0.024431,0.000000,0.087771,2.521005,0.357783,1.085033,0.043410,0.211395,0.086850,0.057731,0.096002,0.020505,0.014031
COL4A1,1.286265,0.090820,0.431724,0.371540,0.076502,0.401456,0.615878,1.210494,0.049996,2.424810,1.210763,0.557918,0.097964,0.461191,3.573218,0.440986,0.099359
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
RHOA,5.429462,3.179174,4.409805,3.888658,2.475262,3.424071,4.800074,4.626585,5.440864,5.019468,4.548037,3.698738,2.462866,3.859793,5.911214,4.529040,2.446507
ROCK1,1.763811,1.142105,2.652488,1.874291,1.572783,0.652349,1.689773,1.391467,1.490019,1.568747,2.507751,1.506479,1.466194,2.130452,3.360094,2.279005,1.864062
ROCK2,1.858847,0.823356,2.050896,2.003272,1.591855,0.446076,1.866784,1.336727,0.904964,1.490392,1.940373,1.233919,1.229125,1.864187,2.150447,1.917266,1.440740
RAC1,6.095888,4.731166,5.611995,5.180840,5.461622,5.906838,5.875602,5.099959,5.859174,5.232840,4.974703,4.769382,4.621963,5.456823,6.045858,5.341306,5.127599


In [21]:
# Percentage of each cell type that express each gene
ct_pct

,Astrocyte,CP,Dorsal Telencephalic IP,Dorsal Telencephalic NPC,Dorsal Telencephalic Neuron,EC,Glioblast,MC,Microglia,NC Derivatives,Neuroepithelium,Non-telencephalic NPC,Non-telencephalic Neuron,OPC,PSC,Ventral Telencephalic NPC,Ventral Telencephalic Neuron
COL1A1,1.292998,3.815560,0.447094,0.629599,0.641790,0.734394,1.220034,33.614759,9.375000,38.890595,7.855904,5.146122,1.824193,1.093093,27.114769,0.935484,0.392836
COL1A2,7.102296,12.641790,0.758116,1.278831,0.392602,0.000000,3.921717,44.580196,16.666667,63.606591,17.394671,11.188838,2.988092,2.414414,49.611340,4.270968,1.961502
COL2A1,0.241033,9.567354,1.762457,8.400748,0.160013,0.203998,0.538324,4.854586,0.000000,12.680381,58.039678,11.629392,1.403123,1.861862,81.618656,19.552688,0.492831
COL3A1,2.404200,2.870077,0.291583,0.232028,0.455912,0.000000,1.738234,32.725834,7.291667,16.856002,0.781991,3.952987,1.614099,1.141141,2.057613,0.458065,0.284806
COL4A1,26.427813,1.812068,8.799326,7.482453,1.565003,8.486332,12.590245,21.768514,1.041667,44.171528,26.513540,10.955399,1.931887,9.489489,69.593050,10.116129,2.012392
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
RHOA,88.681673,56.312488,77.081578,67.311734,44.961608,63.769890,80.142379,80.569480,85.416667,84.484700,79.874358,64.317044,43.621285,69.021021,99.268404,79.559140,43.922647
ROCK1,36.504208,22.360473,50.249465,35.496250,29.431634,13.341493,33.476719,28.009255,28.125000,31.767475,51.487092,29.282144,27.009348,41.189189,74.714220,45.197849,34.229416
ROCK2,38.005556,16.480631,39.571049,37.413157,29.563274,9.179927,36.013634,27.122359,17.708333,29.249821,40.078963,23.967620,22.708615,36.564565,51.714678,38.733333,27.155688
RAC1,93.322575,75.939986,90.196333,82.946843,84.552638,95.389637,90.256081,85.896942,89.583333,86.766963,84.378715,77.586320,73.152635,87.147147,99.497028,85.858065,79.776083


In [ ]:
# QC to flag genes that are expressed in >= 5% of at least one cell type

max_pct_in_any_ct = ct_pct.max(axis=1)
results["max_pct_any_celltype"] = results["gene_symbol"].map(max_pct_in_any_ct)
results["passes_celltype_filter"] = results["max_pct_any_celltype"] >= MIN_PCT_CELLS_CELLTYPE
results["top_celltype"] = ct_means.idxmax(axis=1).reindex(results["gene_symbol"]).values

print(f"  Genes ≥{MIN_PCT_CELLS_CELLTYPE}% in ≥1 cell type: {results['passes_celltype_filter'].sum()}/{len(results)}")




  Genes ≥5.0% in ≥1 cell type: 208/257


### 6. Figuring out cell expressiong per region

In [70]:
regions = adata_ecm.obs[REGION_COL].astype(str)
unique_regions = sorted(regions.unique())
print(f"  Regions: {', '.join(unique_regions)}")

  Regions: Cerebellum, Dorsal midbrain, Dorsal telencephalon, Hypothalamus, Medulla, Pons, Thalamus, Unspecific, Ventral midbrain, Ventral telencephalon


In [104]:
region_means = pd.DataFrame(index=found, columns=unique_regions, dtype=float)
for reg in unique_regions:
    mask = (regions == reg).values
    n_cells = mask.sum()
    if n_cells == 0:
        continue
    X_reg = X[mask, :]
    region_means[reg] = np.array(X_reg.mean(axis=0)).flatten()


results["top_region"] = results["gene_symbol"].map(region_means.idxmax(axis=1))
results.head()

,gene_symbol,pct_cells_expressing,mean expression,is_hvg_in_atlas,Category,Subcategory,Matrisome Division,passes_global_filter,max_pct_any_celltype,passes_celltype_filter,top_celltype,top_region,temporal_cv
246,ACTB,97.233163,7.684915,False,Mechanotransduction,Cytoskeletal,N/A (Intracellular),True,100.000000,True,Microglia,Ventral telencephalon,0.075358
184,RPSA,89.359463,6.517297,False,ECM Receptors,Laminin Receptors,Matrisome-Associated,True,99.862826,True,PSC,Dorsal telencephalon,0.143641
255,RAC1,81.864679,5.163778,False,Mechanotransduction,Rho GTPase Signaling,N/A (Intracellular),True,99.497028,True,Astrocyte,Dorsal telencephalon,0.094649
256,CDC42,73.255061,4.454232,False,Mechanotransduction,Rho GTPase Signaling,N/A (Intracellular),True,97.850937,True,PSC,Ventral telencephalon,0.211264
252,RHOA,58.075386,3.310714,False,Mechanotransduction,Rho GTPase Signaling,N/A (Intracellular),True,99.268404,True,PSC,Unspecific,0.286528


### 7. Figuring out cell expression across time using the organid age column

In [88]:
ages = adata_ecm.obs[AGE_COL]
print(f"  Age range: {ages.min():.0f} - {ages.max():.0f} days")
ages.head()


  Age range: 7 - 450 days


homosapiens_hindbrain_2020_bdrhapsodywholetranscriptomeanalysis_andersenjimena_001_d10_1016_j_cell_2020_11_017_53    45
homosapiens_hindbrain_2020_bdrhapsodywholetranscriptomeanalysis_andersenjimena_001_d10_1016_j_cell_2020_11_017_69    45
homosapiens_hindbrain_2020_bdrhapsodywholetranscriptomeanalysis_andersenjimena_001_d10_1016_j_cell_2020_11_017_72    45
homosapiens_hindbrain_2020_bdrhapsodywholetranscriptomeanalysis_andersenjimena_001_d10_1016_j_cell_2020_11_017_76    45
homosapiens_hindbrain_2020_bdrhapsodywholetranscriptomeanalysis_andersenjimena_001_d10_1016_j_cell_2020_11_017_78    45
Name: organoid_age_days, dtype: int64

In [91]:
# Binning the ages
age_bins = [0, 7, 14, 30, 60, 90, 120, 180, 270, 365, 450]
age_labels = ["0-7d", "7-14d", "14-30d", "30-60d", "60-90d", 
              "90-120d", "120-180d", "180-270d", "270-365d", "365-450d"]

age_binned = pd.cut(ages, bins=age_bins, labels=age_labels, right=True)
age_binned.head()

homosapiens_hindbrain_2020_bdrhapsodywholetranscriptomeanalysis_andersenjimena_001_d10_1016_j_cell_2020_11_017_53    30-60d
homosapiens_hindbrain_2020_bdrhapsodywholetranscriptomeanalysis_andersenjimena_001_d10_1016_j_cell_2020_11_017_69    30-60d
homosapiens_hindbrain_2020_bdrhapsodywholetranscriptomeanalysis_andersenjimena_001_d10_1016_j_cell_2020_11_017_72    30-60d
homosapiens_hindbrain_2020_bdrhapsodywholetranscriptomeanalysis_andersenjimena_001_d10_1016_j_cell_2020_11_017_76    30-60d
homosapiens_hindbrain_2020_bdrhapsodywholetranscriptomeanalysis_andersenjimena_001_d10_1016_j_cell_2020_11_017_78    30-60d
Name: organoid_age_days, dtype: category
Categories (10, object): ['0-7d' < '7-14d' < '14-30d' < '30-60d' ... '120-180d' < '180-270d' < '270-365d' < '365-450d']

In [99]:
temporal_means = pd.DataFrame(index=found, columns = age_labels, dtype=float)
for label in age_labels:
    mask = (age_binned == label).values
    n_cells = mask.sum()
    if n_cells == 0:
        continue
    X_age = X[mask, :]
    temporal_means[label] = np.array(X_age.mean(axis=0)).flatten()
    print(f"    {label}: {n_cells:,} cells")
temporal_means.head()

    0-7d: 18,158 cells
    7-14d: 9,851 cells
    14-30d: 231,565 cells
    30-60d: 612,137 cells
    60-90d: 228,558 cells
    90-120d: 404,681 cells
    120-180d: 219,093 cells
    180-270d: 43,504 cells
    270-365d: 2,985 cells
    365-450d: 46 cells


,0-7d,7-14d,14-30d,30-60d,60-90d,90-120d,120-180d,180-270d,270-365d,365-450d
COL1A1,0.083880,0.387450,0.264514,0.144926,0.278341,0.165687,0.198520,0.018655,0.009853,0.041168
COL1A2,1.038667,0.777536,0.602517,0.273683,0.489920,0.284801,0.246508,0.029808,0.000000,0.259821
COL2A1,4.021841,1.512865,1.676325,0.256856,0.094787,0.028877,0.012504,0.001788,0.024652,0.066066
COL3A1,0.003092,0.008073,0.062314,0.123734,0.267327,0.153985,0.190217,0.002380,0.000000,0.130715
COL4A1,1.523573,2.000342,0.619779,0.243388,0.351212,0.254602,0.495397,0.491463,0.099295,1.403895


In [ ]:
# Computing variance across the bins to measure how much the genes vary over time. 
# High CV measn the genes changes a lot over time
row_means = temporal_means.mean(axis=1)
row_stds = temporal_means.std(axis=1)
temporal_cv = (row_stds / row_means).replace([np.inf, -np.inf], np.nan)
results["temporal_cv"] = results["gene_symbol"].map(temporal_cv)


In [101]:
results.head()

,gene_symbol,pct_cells_expressing,mean expression,is_hvg_in_atlas,Category,Subcategory,Matrisome Division,passes_global_filter,max_pct_any_celltype,passes_celltype_filter,top_celltype,top_region,temporal_cv
246,ACTB,97.233163,7.684915,False,Mechanotransduction,Cytoskeletal,N/A (Intracellular),True,100.000000,True,Microglia,Ventral telencephalon,0.075358
184,RPSA,89.359463,6.517297,False,ECM Receptors,Laminin Receptors,Matrisome-Associated,True,99.862826,True,PSC,Dorsal telencephalon,0.143641
255,RAC1,81.864679,5.163778,False,Mechanotransduction,Rho GTPase Signaling,N/A (Intracellular),True,99.497028,True,Astrocyte,Dorsal telencephalon,0.094649
256,CDC42,73.255061,4.454232,False,Mechanotransduction,Rho GTPase Signaling,N/A (Intracellular),True,97.850937,True,PSC,Ventral telencephalon,0.211264
252,RHOA,58.075386,3.310714,False,Mechanotransduction,Rho GTPase Signaling,N/A (Intracellular),True,99.268404,True,PSC,Unspecific,0.286528


In [ ]:
# grouping genes by ECM production and ECM sesning
# only isolate the genes that were in akansha's paper so that we can find out which cells are expressing these genes in other organoids as well?
# isolate the cells that pass the qc, is there an overlap between the two QCs?


In [105]:
results["temporal_cv"].describe()


count    257.000000
mean       0.990062
std        0.489955
min        0.075358
25%        0.630217
50%        0.888612
75%        1.263624
max        2.707437
Name: temporal_cv, dtype: float64

In [ ]:
# Determining whihc genes are most variale and least variable over time
results.sort_values("temporal_cv", ascending=False).head(10)  # most dynamic
#results.sort_values("temporal_cv").head(10)                    # most stable


,gene_symbol,pct_cells_expressing,mean expression,is_hvg_in_atlas,Category,Subcategory,Matrisome Division,passes_global_filter,max_pct_any_celltype,passes_celltype_filter,top_celltype,top_region,temporal_cv
129,ITGB3,0.185081,0.008882,True,Integrins,Beta Subunits,Matrisome-Associated,False,0.720476,False,MC,Ventral midbrain,2.707437
125,ITGAD,0.010392,0.000513,False,Integrins,Alpha Subunits (Leukocyte),Matrisome-Associated,False,1.041667,False,Microglia,Ventral midbrain,2.373173
123,ITGAM,0.068226,0.003204,False,Integrins,Alpha Subunits (Leukocyte),Matrisome-Associated,False,22.916667,True,Microglia,Ventral telencephalon,2.290196
70,OPTC,0.109512,0.004899,False,Proteoglycans,SLRP,Core Matrisome,False,3.795153,False,PSC,Unspecific,2.289615
24,COL10A1,0.259011,0.012781,False,Collagens,Network,Core Matrisome,False,1.073193,False,Neuroepithelium,Dorsal midbrain,2.265697
98,FBLN5,2.780335,0.140663,True,ECM Glycoproteins,Fibulins,Core Matrisome,True,16.309135,True,MC,Medulla,2.232762
130,ITGB4,1.094784,0.052240,False,Integrins,Beta Subunits,Matrisome-Associated,True,15.129913,True,Astrocyte,Unspecific,2.220888
191,MMP7,0.096918,0.005159,True,ECM Regulators,Matrix Metalloproteinases,Matrisome-Associated,False,0.841572,False,Astrocyte,Unspecific,2.211558
52,LAMB4,0.920434,0.045385,False,Laminins,Beta Chains,Core Matrisome,False,2.615839,False,Non-telencephalic NPC,Ventral midbrain,2.145019
182,CD44,4.686436,0.242587,True,ECM Receptors,Hyaluronan Receptors,Matrisome-Associated,True,39.860283,True,Astrocyte,Unspecific,2.144927


In [136]:
# Summary

results["passes_combined"] = results["passes_global_filter"] & results["passes_celltype_filter"]
results["hvg_passes_combined"] = results["passes_global_filter"] & results["passes_celltype_filter"]& results["is_hvg_in_atlas"]

# Sort by category then by expression
results = results.sort_values(["Category", "pct_cells_expressing"], ascending=[True, False])


In [122]:
results

,gene_symbol,pct_cells_expressing,mean expression,is_hvg_in_atlas,Category,Subcategory,Matrisome Division,passes_global_filter,max_pct_any_celltype,passes_celltype_filter,top_celltype,top_region,temporal_cv,passes_combined,hvg_passes_combined
13,COL6A1,22.252112,1.147462,True,Collagens,Beaded Filament,Core Matrisome,True,68.038409,True,MC,Ventral midbrain,0.320442,True,True
5,COL4A2,15.494827,0.788635,False,Collagens,Basement Membrane / Network,Core Matrisome,True,65.706447,True,PSC,Unspecific,0.710598,True,False
8,COL4A5,13.215176,0.686617,False,Collagens,Basement Membrane / Network,Core Matrisome,True,61.636946,True,PSC,Dorsal midbrain,0.979251,True,False
25,COL11A1,11.951860,0.615274,False,Collagens,Fibrillar,Core Matrisome,True,36.308113,True,Astrocyte,Unspecific,0.737864,True,False
9,COL4A6,10.541755,0.539285,False,Collagens,Basement Membrane / Network,Core Matrisome,True,62.780064,True,PSC,Dorsal midbrain,0.994635,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
63,FMOD,0.170622,0.008555,True,Proteoglycans,SLRP,Core Matrisome,False,4.435299,False,MC,Unspecific,0.801992,False,False
70,OPTC,0.109512,0.004899,False,Proteoglycans,SLRP,Core Matrisome,False,3.795153,False,PSC,Unspecific,2.289615,False,False
69,KERA,0.025698,0.001283,False,Proteoglycans,SLRP,Core Matrisome,False,0.641325,False,MC,Unspecific,1.220547,False,False
71,EPYC,0.024399,0.001225,True,Proteoglycans,SLRP,Core Matrisome,False,0.505348,False,MC,Unspecific,1.135229,False,False


In [133]:
# Print summary by category
print(f"\n{'Category':<30} {'Total':<8} {'In Atlas':<10} {'Pass Global':<12} {'Pass CT':<10} {'Pass Both':<10} {'HVG':<6} {'HVG and Pass Both:'}")
print("-" * 120)
for cat in results["Category"].unique():
    cat_mask = results["Category"] == cat
    sub = results[cat_mask]
    total_curated = ecm_df[ecm_df["Category"] == cat]["Gene Symbol"].nunique()
    print(f"{cat:<30} {total_curated:<8} {len(sub):<10} {sub['passes_global_filter'].sum():<12} "
          f"{sub['passes_celltype_filter'].sum():<10} {sub['passes_combined'].sum():<10} {sub['is_hvg_in_atlas'].sum():<10}"
          f"{sub['hvg_passes_combined'].sum()}")

total_pass = results["passes_combined"].sum()
print(f"\n  TOTAL genes passing combined filter: {total_pass}/{len(results)}")
print(f"  Of which are also HVGs: {(results['passes_combined'] & results['is_hvg_in_atlas']).sum()}")




Category                       Total    In Atlas   Pass Global  Pass CT    Pass Both  HVG    HVG and Pass Both:
------------------------------------------------------------------------------------------------------------------------
Collagens                      44       44         28           34         27         17        14
ECM Glycoproteins              27       27         23           25         23         14        13
ECM Receptors                  8        8          7            8          7          2         2
ECM Regulators                 36       36         23           23         21         11        6
ECM-Associated Signaling       7        7          7            7          7          2         2
Integrins                      26       26         14           21         13         5         1
Laminins                       12       12         7            9          7          4         3
Matricellular Proteins         6        6          2            2          2  

In [145]:
# Which are the mechanotransduction genes that are highly variable

mt_hvg = results[(results["hvg_passes_combined"]) & (results["Category"] == "Mechanotransduction") ]["gene_symbol"].tolist()
mt_hvg

['SYNE2', 'LMNB1', 'LMNA', 'ACTA2']

In [148]:
results[results["gene_symbol"].isin(mt_hvg)]

,gene_symbol,pct_cells_expressing,mean expression,is_hvg_in_atlas,Category,Subcategory,Matrisome Division,passes_global_filter,max_pct_any_celltype,passes_celltype_filter,top_celltype,top_region,temporal_cv,passes_combined,hvg_passes_combined
242,SYNE2,47.201535,2.795328,True,Mechanotransduction,LINC Complex,N/A (Nuclear Envelope),True,92.943692,True,Dorsal Telencephalic IP,Dorsal telencephalon,0.323128,True,True
238,LMNB1,26.028901,1.393774,True,Mechanotransduction,Nuclear Mechanotransduction,N/A (Nuclear),True,71.509104,True,Dorsal Telencephalic IP,Ventral telencephalon,0.656701,True,True
237,LMNA,23.646515,1.222906,True,Mechanotransduction,Nuclear Mechanotransduction,N/A (Nuclear),True,68.867056,True,NC Derivatives,Unspecific,0.452511,True,True
245,ACTA2,4.241779,0.217199,True,Mechanotransduction,Cytoskeletal,N/A (Intracellular),True,68.665851,True,EC,Medulla,1.263624,True,True


In [149]:
# From these results perhaps we are seeing that mecahnotrasnduction appears to happen in the dorsal telencephalon and the ventral telencephalon. It would be interesting to see the relevant time points